# Data Cleaning - Stack Overflow Developer Survey (2023-2025)
## Objetive:
- Get the 3 years of survey data ready to compare focused on the main question (check the README)

This notebook does not explore every column, 
so it works from a scope that we already defined in `docs/schema_crosswalk.md` and it is based on what fields are safe to compare
across years. The real analysis usually starts with a hypothesis, so it does not start with a blind scan of every column.


In [1]:
import pandas as pd
# Load raw survey data
df_2023 = pd.read_csv("../data/raw/results_2023.csv")
df_2024 = pd.read_csv("../data/raw/results_2024.csv")
df_2025 = pd.read_csv("../data/raw/results_2025.csv", low_memory=False)  # mixed types warning
print(df_2023.shape)
print(df_2024.shape)
print(df_2025.shape)

(89184, 84)
(65437, 114)
(49191, 172)


Shapes match what we counted in the schema crosswalk (84, 114, 172 columns), so it confirms that the files are the right ones.

### Role name consistency

The survey renames some roles across years, we mentioned when we were planning (check `docs/schema_crosswalk.md`). Now, we need to check 
which roles changed before comparing them.

In [2]:
# Check DevType naming across years
print(df_2023["DevType"].unique())
print(df_2024["DevType"].unique())
print(df_2025["DevType"].unique())

<StringArray>
[                                            nan,
          'Senior Executive (C-Suite, VP, etc.)',
                           'Developer, back-end',
                          'Developer, front-end',
                         'Developer, full-stack',
                          'System administrator',
 'Developer, desktop or enterprise applications',
                         'Developer, QA or test',
                                      'Designer',
 'Data scientist or machine learning specialist',
                      'Data or business analyst',
                         'Security professional',
                                      'Educator',
                   'Research & Development role',
                       'Other (please specify):',
                             'Developer, mobile',
                        'Database administrator',
   'Developer, embedded applications or devices',
                                       'Student',
                                'Eng

#### Findings:
- Data Analyst: "Data or business analyst" in all 3 years
- Data Scientist: "Data scientist or machine learning
  specialist" (2023-2024) vs "Data scientist" (2025)
- Data Engineer: "Engineer, data" (2023) vs "Data engineer" (2024-2025)
- DBA: "Database administrator" (2023-2024) vs "Database administrator or
  engineer" (2025)

### Normalize role names

Unify the 4 roles that it changed wording across years in one clean label each, so we can group by role properly.

In [5]:
# Unify role names across years
role_map = {
    "Data or business analyst": "Data Analyst",
    "Data scientist or machine learning specialist": "Data Scientist",
    "Data scientist": "Data Scientist",
    "Engineer, data": "Data Engineer",
    "Data engineer": "Data Engineer",
    "Database administrator": "Database Administrator",
    "Database administrator or engineer": "Database Administrator",
}

# Mapping: Roles not in the dict = NaN
df_2023["RoleClean"] = df_2023["DevType"].map(role_map)
df_2024["RoleClean"] = df_2024["DevType"].map(role_map)
df_2025["RoleClean"] = df_2025["DevType"].map(role_map)

# Count responses per role
print(df_2023["RoleClean"].value_counts())
print(df_2024["RoleClean"].value_counts())
print(df_2025["RoleClean"].value_counts())

RoleClean
Data Scientist            1588
Data Engineer             1248
Data Analyst               837
Database Administrator     257
Name: count, dtype: int64
RoleClean
Data Engineer             1118
Data Scientist            1024
Data Analyst               523
Database Administrator     171
Name: count, dtype: int64
RoleClean
Data Engineer             770
Data Scientist            574
Data Analyst              351
Database Administrator    175
Name: count, dtype: int64


### Check sample size - AIComplex and AIAgents by role

- AIComplex only exists in 2024 and 2025
- AIAgents only exists in 2025

Then, we check if each role has enough responses to trust the numbers.

In [7]:
# Count AIComplex responses per role (2024 - 2025)
print(df_2024.groupby("RoleClean")["AIComplex"].count())
print(df_2025.groupby("RoleClean")["AIComplex"].count())

# Count AIAgents responses per role (2025)
print(df_2025.groupby("RoleClean")["AIAgents"].count())

RoleClean
Data Analyst              290
Data Engineer             654
Data Scientist            719
Database Administrator     70
Name: AIComplex, dtype: int64
RoleClean
Data Analyst              253
Data Engineer             587
Data Scientist            447
Database Administrator    134
Name: AIComplex, dtype: int64
RoleClean
Data Analyst              240
Data Engineer             560
Data Scientist            429
Database Administrator    126
Name: AIAgents, dtype: int64


## Verdict

**Data confirmed workable**

- Role names unified with `role_map`
- Sample sizes are enough for all 4 roles

**Risk Found**: `AIComplex` for Database Administrator in 2024 has only n=70, so a percentage alone could mislead and a few responses change it a
lot. 

**Fix**: Show raw count next to the percentage in that chart.
